In [0]:
from pyspark.sql.functions import current_timestamp, col

def get_new_files(volume_path,processed_table):
    folders=dbutils.fs.ls(volume_path)
    new_files=[]
    for folder in folders:
        files=dbutils.fs.ls(folder.path)
        for file in files:
            new_files.append({
            'file_name':file.name,
            'file_path':file.path,
             'source_name':folder.name
            })

    new_files_df=spark.createDataFrame(new_files)
    new_files_df.display()


    processed_files=spark.table(processed_table)\
    .select('source_name','file_name')
    processed_files.display()

    new_files_df=new_files_df.join(processed_files,['source_name','file_name'],'left_anti')
    new_files_df.display()
    return new_files_df



In [0]:
import configparser
config=configparser.ConfigParser()
config.read('/Workspace/Users/krishraj2408@gmail.com/Healthcare_Fraud_detection_pipeline/config2.ini')

In [0]:
# new_files_df=get_new_files(config['VOLUME']['volume_path'],config['TABLES']['processed_file_table'])

In [0]:
def get_table_name(config,source_name):
    if('claims' in source_name):
      table_name=config["TABLES"][source_name.replace("_claims/", "")+'_table']
      return table_name
    else:
        table_name=config["TABLES"][source_name.replace("/", "")+'_table']
        return table_name
    print(table_name)



In [0]:
METADATA_COLUMNS = {
    "ingestion_ts",
    "created_ts",
    "updated_ts",
    "batch_id",
    "source_file"
}

def validate_columns(table_name,file_path):
    expected_columns=spark.table(table_name).columns
    incoming_columns=(
         spark.read
         .option("header", True)
         .csv(file_path)
         .columns
        )  
    print(incoming_columns)
    

    expected_columns=[c.strip().lower() for c in expected_columns if c not in METADATA_COLUMNS]
    incoming_columns=[c.strip().lower() for c in incoming_columns ]
    if(expected_columns!=incoming_columns):
        missing_columns=list(set(expected_columns)-set(incoming_columns))
        extra_columns=list(set(incoming_columns)-set(expected_columns))
        print(f"Extra columns: {extra_columns}")
        print(f"Missing columns: {missing_columns}")
        raise Exception(
            f"""
          Header validation failed.

           File: {file_path}
          Target Table: {table_name}

         Expected Columns:
         {expected_columns}

         Incoming Columns:
         {incoming_columns}

          Missing Columns:
          {missing_columns}

          Extra Columns:
          {extra_columns}
          """
        )

    return True


    


In [0]:
def load_incremental_file(file_path,table_name):
    try:
      print("Step 1: Validate")
      validate_columns(table_name,file_path)

      print("Step 2: Read schema")
      schema=spark.table(table_name).schema
      
      print("Step 3: Read csv")
      return ( spark.read
        .format("csv")
        .option("header", True)
        .schema(schema)
        .load(file_path)
        .select([col(c) for c in schema.fieldNames()])
        )
    except Exception as e:
        print(e)
        raise Exception(
            f"Failed to load {file_path} into {table_name}: {e}"
        )

In [0]:
def write_bronze(df, table_name):
    (df.withColumn('ingestion_ts',current_timestamp())\
               .write\
               .format('delta')\
               .mode('append')\
               .saveAsTable(table_name))



In [0]:
def update_file_tracker(file_data):
    tracker_df = spark.createDataFrame([
        {
            "source_name": file_data["source_name"],
            "file_name": file_data["file_name"],
            "file_path": file_data["file_path"],
            "status": "processed"
        }
    ])

    (
        tracker_df
        .withColumn(
            "processed_ts",
            current_timestamp()
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable("healthcare_catalog.metadata.file_tracker")
    )
    

In [0]:
from pyspark.sql.functions import current_timestamp, col

def log_ingestion(
    source_name,
    table_name,
    file_name,
    file_path,
    status,
    records_loaded,
    error_message
):
    log_df=spark.createDataFrame(
        [
            {
                    "source_name": source_name,
                    "table_name": table_name,
                    "file_name": file_name,
                    "file_path": file_path,
                    "status": status,
                    "records_loaded": records_loaded,
                    "error_message": error_message,
                    
            }
        ]
    )

    (log_df.withColumn('ingestion_ts',current_timestamp())\
               .write\
               .format('delta')\
               .mode('append')\
               .saveAsTable("healthcare_catalog.metadata.ingestion_log"))